Analysis of Ethereum CL's Attestation Inclusion metrics using the networking events as reference.  

In [ ]:
import polars as pl
from loaders import load_parquet
from IPython.display import display

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from queries.slot_tags import TAG_ORDERS, TAG_LABELS, TAG_GROUPS, short_label

# Global Variables
target_date = None # Use this as a default for the automation and the rendering of the page

In [ ]:
# read the parquet file
df = pl.from_pandas(load_parquet("slot_tags", target_date=target_date))

# Tag Distribution Overview — blocks, attestations, aggregations

## Tag Distribution Overview

Percentage of slots with each tag value, broken down per dimension and grouped by category (blocks, attestations, aggregations).

In [ ]:
total = len(df)

def plot_tag_distribution(tag_cols: list[str], title: str) -> None:
    n = len(tag_cols)
    row_height = 300
    fig = make_subplots(
        rows=n,
        cols=1,
        subplot_titles=[TAG_LABELS[c] for c in tag_cols],
        vertical_spacing=80 / (row_height * n),  # fixed 80px gap between subplots
    )
    for i, col in enumerate(tag_cols, 1):
        present = set(df[col].unique().to_list())
        order = [v for v in TAG_ORDERS[col] if v in present]
        counts = (
            df
            .group_by(col)
            .agg(count=pl.len())
            .with_columns(pct=(pl.col("count") * 100 / total))
            .with_columns(pl.col(col).cast(pl.Enum(order)))
            .sort(col)
        )
        pct_values = counts["pct"].to_list()
        fig.add_trace(
            go.Bar(
                x=[short_label(v) for v in counts[col].to_list()],
                y=pct_values,
                text=[f"{v:.1f}%" for v in pct_values],
                textposition="outside",
                showlegend=False,
                marker_color="#6366f1",
            ),
            row=i,
            col=1,
        )
        # Extend y-axis range so outside labels stay inside the plot area,
        # preventing the hover tooltip from disappearing when the cursor is over them.
        fig.update_yaxes(title_text="% slots", range=[0, max(pct_values) * 1.25], row=i, col=1)
    fig.update_layout(
        title=title,
        height=row_height * n,
        width=1000,
        margin=dict(t=80, b=40),
        hovermode="x",
    )
    fig.show()


for group_name, cols in TAG_GROUPS.items():
    plot_tag_distribution(cols, f"Slot Tag Distributions — {group_name}")

## Cross-Dimension Co-occurrence Heatmaps

For each pair of dimensions, cells show the percentage of slots in row-category X that also have column-category Y. Rows sum to 100%.

In [ ]:
def cross_tab_heatmap(df: pl.DataFrame, col_x: str, col_y: str) -> go.Figure:
    """Row-normalized cross-tab: for each value of col_x, % breakdown by col_y."""
    present_x = set(df[col_x].unique().to_list())
    present_y = set(df[col_y].unique().to_list())
    order_x = [v for v in TAG_ORDERS[col_x] if v in present_x]
    order_y = [v for v in TAG_ORDERS[col_y] if v in present_y]

    ct = df.group_by([col_x, col_y]).agg(count=pl.len())
    totals = ct.group_by(col_x).agg(total=pl.col("count").sum())
    ct = ct.join(totals, on=col_x).with_columns(pct=(pl.col("count") * 100 / pl.col("total")))

    matrix = []
    for x in order_x:
        row = []
        for y in order_y:
            cell = ct.filter((pl.col(col_x) == x) & (pl.col(col_y) == y))
            row.append(round(cell["pct"][0], 1) if len(cell) > 0 else 0.0)
        matrix.append(row)

    fig = go.Figure(go.Heatmap(
        z=matrix,
        x=[short_label(v) for v in order_y],
        y=[short_label(v) for v in order_x],
        colorscale="Blues",
        zmin=0,
        zmax=100,
        text=[[f"{v:.1f}%" for v in row] for row in matrix],
        texttemplate="%{text}",
        hoverongaps=False,
        colorbar=dict(title="% of row"),
    ))
    fig.update_layout(
        title=f"{TAG_LABELS[col_x]} vs {TAG_LABELS[col_y]}",
        xaxis_title=TAG_LABELS[col_y],
        yaxis_title=TAG_LABELS[col_x],
        width=900,
        height=350,
        margin=dict(l=200, b=140),
    )
    return fig


for col_x, col_y in [
    ("block_proposal_tag", "block_p50_spread_tag"),
    ("block_size_tag",     "block_p50_spread_tag"),
    ("blob_count_tag",     "block_p50_spread_tag"),
    ("block_proposal_tag", "block_p50_arrival_tag"),
]:
    cross_tab_heatmap(df, col_x, col_y).show()

for col_x, col_y in [
    ("block_proposal_tag",    "col_first_seen_p50_tag"),
    ("blob_count_tag",        "col_first_seen_p50_tag"),
    ("col_first_seen_p50_tag", "col_spread_p50_tag"),
]:
    cross_tab_heatmap(df, col_x, col_y).show()

for col_x, col_y in [
    ("block_proposal_tag",     "att_first_seen_p50_tag"),
    ("att_first_seen_p50_tag", "att_spread_p50_tag"),
    ("att_first_seen_p50_tag", "att_inclusion_p50_tag"),
    ("att_spread_p50_tag",     "att_inclusion_p50_tag"),
]:
    cross_tab_heatmap(df, col_x, col_y).show()

for col_x, col_y in [
    ("block_proposal_tag",     "agg_first_seen_p50_tag"),
    ("agg_first_seen_p50_tag", "agg_spread_p50_tag"),
    ("att_first_seen_p50_tag", "agg_first_seen_p50_tag"),
]:
    cross_tab_heatmap(df, col_x, col_y).show()

## Slot Tag Flow (Sankey)

Shows how slots flow across dimensions: **Blob Count → Block Size → Proposal Timing → Broadcast Speed**. Width of each band is proportional to the number of slots.

In [ ]:
def build_sankey(df: pl.DataFrame, cols: list[str], title: str = "") -> go.Figure:
    node_labels: list[str] = []
    node_index: dict[tuple[str, str], int] = {}

    for col in cols:
        present = set(df[col].unique().to_list())
        for val in TAG_ORDERS[col]:
            if val in present:
                node_index[(col, val)] = len(node_labels)
                node_labels.append(f"{TAG_LABELS[col]}: {short_label(val)}")

    sources, targets, values = [], [], []

    for i in range(len(cols) - 1):
        col_a, col_b = cols[i], cols[i + 1]
        flow = df.group_by([col_a, col_b]).agg(count=pl.len())
        for row in flow.iter_rows(named=True):
            a = node_index.get((col_a, row[col_a]))
            b = node_index.get((col_b, row[col_b]))
            if a is not None and b is not None:
                sources.append(a)
                targets.append(b)
                values.append(row["count"])

    fig = go.Figure(go.Sankey(
        arrangement="snap",
        node=dict(
            label=node_labels,
            pad=20,
            thickness=18,
            color="#6366f1",
        ),
        link=dict(
            source=sources,
            target=targets,
            value=values,
            color="rgba(99,102,241,0.25)",
        ),
    ))
    fig.update_layout(
        title=title,
        width=1200,
        height=700,
    )
    return fig


build_sankey(
    df,
    ["block_proposal_tag", "block_size_tag", "blob_count_tag", "block_p50_arrival_tag", "block_p50_spread_tag"],
    title="Slot Tag Flow: Blob Count → Block Size → Proposal Timing → Arrival → Broadcast Speed",
).show()

build_sankey(
    df,
    ["block_proposal_tag", "blob_count_tag", "col_first_seen_p50_tag", "col_spread_p50_tag"],
    title="Data Column Tag Flow (P50): Block Proposal → Blob Count → First Seen → Spread",
).show()

build_sankey(
    df,
    ["block_proposal_tag", "att_first_seen_p50_tag", "att_spread_p50_tag", "att_inclusion_p50_tag"],
    title="Attestation Tag Flow (P50): Block Proposal → First Seen → Spread → Inclusion Delay",
).show()

build_sankey(
    df,
    ["block_proposal_tag", "agg_first_seen_p50_tag", "agg_spread_p50_tag"],
    title="Aggregation Tag Flow (P50): Block Proposal → First Seen → Spread",
).show()